# 11 Role Intelligence
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Construct master role competency profiles linking internal job roles to O*NET essential foundational skills and in-demand software/tools.


In [2]:
import os
import pandas as pd

DATA_PROCESSED = "../data/processed"
df_mapping = pd.read_csv(os.path.join(DATA_PROCESSED, "role_taxonomy_mapping.csv"))
df_occ = pd.read_csv(os.path.join(DATA_PROCESSED, "occupation_master.csv"))
df_ess = pd.read_csv(os.path.join(DATA_PROCESSED, "essential_skills_processed.csv"))
df_soft = pd.read_csv(os.path.join(DATA_PROCESSED, "software_skills_processed.csv"))

print(f"Loaded {len(df_mapping)} role mappings and O*NET skill taxonomies.")


Loaded 23 role mappings and O*NET skill taxonomies.


In [3]:
# Extract top software tools + essential skills per internal job role
role_profiles = []

for _, row in df_mapping.iterrows():
    role = row['job_role']
    soc = row['soc_code']
    onet_title = row['onet_title']
    
    # Essential foundational skills (Importance >= 3.0)
    ess_skills = df_ess[(df_ess['soc_code'] == soc) & (df_ess['importance'] >= 3.0)]['skill_name'].tolist()
    
    # Software & Tech Tools (Prioritize In-Demand / Hot Technology, top 12)
    soft_df = df_soft[df_soft['soc_code'] == soc].sort_values(
        by=['in_demand', 'hot_technology'], ascending=False
    )
    soft_skills = soft_df['skill_name'].head(12).tolist()
    
    all_required = list(dict.fromkeys(ess_skills + soft_skills))
    
    role_profiles.append({
        'job_role': role,
        'soc_code': soc,
        'onet_title': onet_title,
        'essential_skills_count': len(ess_skills),
        'software_skills_count': len(soft_skills),
        'total_required_skills': len(all_required),
        'required_skills_list': "|".join(all_required)
    })

df_role_profiles = pd.DataFrame(role_profiles)
out_role_prof = os.path.join(DATA_PROCESSED, "role_competency_profiles.csv")
df_role_profiles.to_csv(out_role_prof, index=False)
print("=== ROLE COMPETENCY PROFILES (Sample 5) ===")
print(df_role_profiles[['job_role', 'soc_code', 'total_required_skills']].head().to_string(index=False))


=== ROLE COMPETENCY PROFILES (Sample 5) ===
                 job_role   soc_code  total_required_skills
          Sales Executive 41-4012.00                     19
       Research Scientist 15-1221.00                     22
    Laboratory Technician 29-2012.00                     20
   Manufacturing Director 11-1021.00                     20
Healthcare Representative 41-4011.00                     19
